This notebook is designed to determine the base window for neutron detection.
We use data from a reference experiment with sufficient neutron signal (currently ID-419).
The PSD/Energy data is then put into a 2D histogram, with energy cuts approximately every 15 keVee, and X total PSD cuts.
This 2D histogram is split using the energy cuts into a series of 1D histogram "energy" slices.
Each slice is modeled on a bimodal distribution of PSD vs. count, with the lower PSD Gaussian being for gamma ray events, and the higher PSD Gaussian being for neutrons.
Using these fit values, the FOM value is also calculated as abs(mu_n-mu_g)/(2.56*(sigma_n+sigma_g))
These values are used to define the neutron window as follows:

- Left boundary: Find energy A where FOM passes 1.27
- Right boundary: E = 688 keVee (Compton scattering edge, adjusted by detector energy resolution)
- Bottom boundary: For each slice, get point where x = slice energy midpoint, y = mu_n - 3 * sigma_n, connect points
- Top boundary: For each slice, get point where x = slice energy midpoint, y = mu_n + 3 * sigma_n, connect points

## Initialization

In [ ]:
# Importing needed code

from typing import Callable, TypeVar, Any
from datetime import timedelta
from math import sqrt, log
import pickle

import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
import pandas as pd
import numpy as np

from data_processing.arc_paths import (
    get_parq_root, get_report_root, get_exp_root, INPUT_DATA_FOLDER)
from data_processing.dataframe_validation import DataframeColumn
from data_processing.loading.dataframe_loading import load_psd
from data_processing.loading.timetag_processing import calculate_timetag_hours
from data_processing.processing.bimodal_fitting import (
    get_psd_energy_histogram,
    scan_histogram_slices,
    BimodalBounds,
    BimodalParams
)
from data_processing.processing.neutron_classification import (
    generate_new_neutron_window,
    classify
)
from data_processing.reporting.plotting import plot_scatter, plot_classification
# from data_processing.processing.figure_of_merit import (
#     FOM,
#     gaussian,
#     bimodal
# )
from data_processing.helpers.stop_jupyter import stop

In [ ]:
# Constants
SMOOTHING_WINDOW_SIZE = 5

In [ ]:
T = TypeVar('T')


def get_input_with_default(
    prompt: str, default: int, converter: Callable[[str], T]
) -> int:
    raw_value = input(prompt)
    try:
        value = converter(raw_value)
    except ValueError:
        value = default
    return value

In [ ]:
def get_window_size(data_type: str) -> int:
    prompt = (f"Enter smoothing window size for {data_type}, "
              + "or press Enter for default (5):")
    return get_input_with_default(prompt, SMOOTHING_WINDOW_SIZE, int)


def load_non_neutron_data(exp_name, file_names):
    if not isinstance(file_names, list):
        file_names = [file_names]
    for file_name in file_names:
        file_path = get_exp_root(exp_name) / file_name
        if file_path.is_file():
            df = pd.read_csv(file_path)
            return df
    return None


def smooth_non_neutron_data(
    df, raw_data_col_name, smoothed_data_col_name, window_size
):
    df[smoothed_data_col_name] = df[raw_data_col_name].rolling(
        window=window_size, min_periods=1).mean()
    return df


def localize_time(df, time_column):
    local_tz = 'America/Vancouver'
    naive_time = pd.to_datetime(df[time_column])
    try:
        localized_time = naive_time.dt.tz_localize(local_tz)
    except TypeError:
        localized_time = naive_time.dt.tz_convert(local_tz)
    df[time_column] = localized_time
    return df

In [ ]:
def bin_non_neutron_data(df, time_bins, data_col, selected_cols):
    start_time = time_bins[0]
    df = get_time_cut(df, 'Time', time_bins)

    binned_df = df.groupby("Time Bin", as_index=False)[data_col] \
        .agg(['mean', 'std']) \
        .copy()
    binned_df.columns = selected_cols
    binned_df['Bin midpoint'] = binned_df.index.to_series() \
        .apply(lambda x: x.mid)
    binned_df = bin_midpoint_time_to_seconds(binned_df, start_time)

    return binned_df

In [ ]:
def bin_midpoint_time_to_seconds(df, start_time):
    zeroed_midpoint = pd.to_datetime(df["Bin midpoint"]) - start_time
    df['Bin time (s)'] = zeroed_midpoint.dt.total_seconds()
    return df

In [ ]:
def get_time_cut(df, time_tag_col, time_bins):
    timetag_cut = pd.cut(df[time_tag_col], bins=time_bins)
    df['Time Bin'] = timetag_cut
    return df

In [ ]:
def get_endings_with_modifications(endings, string_method):
    modded_endings = [
        *endings,
        *[getattr(ending, string_method)() for ending in endings]
    ]
    return modded_endings


def get_possible_names(exp_name, endings):
    possible_names = [f"{exp_name}{ending}.csv"
                      for ending in possible_filename_endings]
    return possible_names


def get_filenames_with_hyphenless_ids(exp_name, filenames):
    hyphenless = exp_name.replace('-', '')
    modded_filenames = [
        *filenames,
        *[filename.replace(exp_name, hyphenless) for filename in filenames]
    ]
    return modded_filenames


def get_full_possible_names_set(exp_name, endings):
    possible_names = get_possible_names(exp_name, endings)
    full_poss_names_set = get_filenames_with_hyphenless_ids(
        exp_name, possible_names)
    return full_poss_names_set

## Experiment ID Input

In [ ]:
experiment_id = "ID-419"
id_valid = get_parq_root(experiment_id).is_dir()
if not id_valid:
    print(f"Experiment {experiment_id} cannot be found")
    stop()
# done = False
# experiment_ids = ["ID-213"]
# while not done:
#     ids_valid = []
#     for exp_id in experiment_ids:
#         id_valid = get_parq_root(exp_id).is_dir()
#         ids_valid.append(id_valid)
#         if not id_valid:
#             print(f"Experiment {exp_id} cannot be found")

#     done = all(ids_valid)
#     if not done:
#         print("Invalid experiment IDs, please fix")
#         experiment_ids = []
#     else:
#         print("All experiment IDs are valid")

## Data Loading and Initial Processing

### Neutron Data Processing

In [ ]:
# Data Loading
exp_data: dict[str, Any] = {'unclassified': load_psd(experiment_id)}

In [ ]:
# Express timetags in hours elapsed
exp_data['unclassified'] = calculate_timetag_hours(exp_data['unclassified'])

In [ ]:
# Get PSD/Energy 2D histogram
start_scan_idx = 0
end_scan_idx = 420

psd_report: pd.DataFrame = exp_data['unclassified']

Z, xe, ye = get_psd_energy_histogram(psd_report)
end_scan_idx = min(end_scan_idx, len(Z))

exp_data['psd_histogram'] = Z
exp_data['histogram_x_edges'] = xe[start_scan_idx:end_scan_idx+1]
exp_data['histogram_y_edges'] = ye

In [ ]:
# Scan energy slices, get bimodal fits
nan_total_threshold = 5
nan_window_threshold = 4
nan_rolling_window = 7
stop_here = False

Z = exp_data['psd_histogram']
xe = exp_data['histogram_x_edges']
ye = exp_data['histogram_y_edges']

# Default
default_bounds: BimodalBounds = (
    BimodalParams(0.1, 0.01, 1,
        0.25, 0.01, 0),
    BimodalParams(0.2, 0.1, Z.max(),
        0.38, 0.04, 2000)
)

bounds_a: BimodalBounds = (
    BimodalParams(0.1, 0.01, 1,
        0.35, 0.01, 0),
    BimodalParams(0.2, 0.1, Z.max(),
        0.36, 0.04, 2000)
)

bounds_b: BimodalBounds = (
    BimodalParams(0.1, 0.01, 1,
        0.34, 0.01, 0),
    BimodalParams(0.2, 0.1, Z.max(),
        0.36, 0.03, 2000)
)

# Ranged Example
bounds = [
    ((0, 60), bounds_a),
]

psd_bin_lbs = ye[:-1]
psd_bin_ubs = ye[1:]
psd_bin_centers = [(lb+ub)/2 for lb, ub in zip(psd_bin_lbs, psd_bin_ubs)]
all_slice_xs = xe[:end_scan_idx]

df, df_err = scan_histogram_slices(
    psd_bin_centers, 
    Z.T, 
    all_slice_xs, 
    default_bounds, 
    bounds=bounds, 
    start_idx=start_scan_idx, 
    end_idx=end_scan_idx)
nan_rows = df.isna().any(axis=1)
nan_rows = nan_rows[nan_rows]
if nan_rows.shape[0] > 0:
    nan_indexes = np.where(nan_rows)[0]
    total_nan_rows = len(nan_indexes)
    rolling_nan_count = nan_rows.rolling(window=nan_rolling_window) \
        .sum() \
        .max()

    print(f"Fit issues in {experiment_id}")
    print(f"Fit failed on following slice indexes: {nan_indexes}")

    if (total_nan_rows > nan_total_threshold
            or rolling_nan_count > nan_window_threshold):
        print(f"Experiment {experiment_id} could not be classified")
        print(f"Total failed slices: {total_nan_rows}")
        print(
            f"Max failed slices in a {nan_rolling_window} slice window:"+
            f" {rolling_nan_count}"
        )

        exp_data['valid_slice_fits'] = df.dropna().copy()
        exp_data['bad_slice_indexes'] = nan_indexes

        stop_here = True

    # filter out all nan rows from df
    df = df.dropna().copy()
    # continue as normal to try fitting with bad rows ignored
exp_data['fom_results'] = df

if stop_here:
    stop()

In [ ]:
# Generate neutron window
sigma = 3  # Hey y'all, here's where you change sigma!

df = exp_data['fom_results']

borders = generate_new_neutron_window(
    df,
    all_slice_xs,
    sigma=sigma
)
exp_data["n_window_borders"] = borders

# classified_df = classify(
#     psd_report,
#     neutron_lb_fit,
#     neutron_ub_fit,
#     DataframeColumn.NEUTRON_CLASS,
#     le_cutoff=L0
# )

# exp_data["psd_report"] = classified_df
# experiment_neutron_data[exp_name] = exp_data

In [ ]:
# classify neutrons
psd_report = exp_data['unclassified']
borders = exp_data['n_window_borders']
classified_df = classify(psd_report, borders, DataframeColumn.NEW_N_CLASS)
exp_data['psd_report'] = classified_df    

## Export and Display

In [ ]:
# Plot classification
HISTOGRAM_RES = 1024
COUNT_LIMIT = 20

psd_report = exp_data['psd_report']
borders = exp_data['n_window_borders']
fig, ax = plot_classification(
    psd_report, 
    borders,
    experiment_id, 
    DataframeColumn.NEW_N_CLASS, 
    count_limit=COUNT_LIMIT, 
    colormap_name = "seismic"
)

plt.show()

In [ ]:
# Save window boundaries
save_folder = INPUT_DATA_FOLDER / "ReferenceWindow"
save_folder.mkdir(parents=True, exist_ok=True)
borders = exp_data['n_window_borders']

left_border = borders.left
right_border = borders.right
save_file_path = save_folder / "side_borders.txt"
with save_file_path.open('w') as save_file:
    save_file.writelines(
        [
            f"left: {left_border if left_border is not None else 'None'}", 
            f"right: {right_border if right_border is not None else 'None'}"
        ]
    )
print(f"Saved side borders to {save_file_path}")

top_border = borders.top
if top_border is not None:
    top_border_file_path = save_folder / "top_border.pkl"
    with top_border_file_path.open('wb') as top_file:
        pickle.dump(top_border, top_file)
else:
    print("No top border")

bottom_border = borders.bottom
if bottom_border is not None:
    bottom_border_file_path = save_folder / "bottom_border.pkl"
    with bottom_border_file_path.open('wb') as bottom_file:
        pickle.dump(bottom_border, bottom_file)
else:
    print("No bottom border")

In [ ]:
input("Processing done, hit Enter to finish")
stop()